In [1]:
import pandas as pd
import json
import os
from pathlib import Path

def parse_info_csv(csv_path):
    df = pd.read_csv(csv_path)
    
    # Создаем списки для новых данных
    image_names = []
    bboxes = []
    labels = []
    image_paths = []
    
    # Обрабатываем каждую строку
    for _, row in df.iterrows():
        # Парсим JSON с аннотациями
        annotations = json.loads(row['box_and_label'])
        
        # Для каждого бокса в аннотациях создаем отдельную строку
        for ann in annotations[0]:
            image_names.append(row['image_name'])
            bboxes.append([
                ann['left'],
                ann['top'],
                ann['width'],
                ann['height']
            ])
            labels.append(ann['label'].replace('\n', ' '))  # Заменяем переносы строк на пробелы
            image_paths.append(str(Path('dataset2_2') / 'test' / row['image_path']))
    
    # Создаем новый DataFrame
    new_df = pd.DataFrame({
        'image_name': image_names,
        'bbox': bboxes,
        'label': labels,
        'image_path': image_paths
    })
    
    return new_df

# Определяем базовый путь
base_path = Path.cwd()  # Или укажите явный путь: Path('/путь/к/вашей/папке')

# Путь к файлу test info.csv
test_csv_path = base_path / 'dataset2_2' / 'test' / 'info.csv'

# Проверяем существование файла
print(f"Test CSV exists: {test_csv_path.exists()}")

if test_csv_path.exists():
    # Создаем DataFrame для test
    test_df = parse_info_csv(test_csv_path)

    # Выводим информацию о DataFrame
    print("\nTest DataFrame:")
    print(test_df.head())

    # Сохраняем DataFrame в файл (опционально)
    test_df.to_csv(base_path / 'd2_2_test.csv', index=False) 
    
else:
    print("\nОшибка: Не удалось найти CSV файл.")
    print("Проверьте структуру папок:")
    print(f"Ожидаемый путь: {test_csv_path}")
    print("\nТекущая структура папок:")
    for root, dirs, files in os.walk(base_path):
        print(f"Папка: {root}")
        for file in files:
            print(f"  - {file}")

Test CSV exists: True

Test DataFrame:
  image_name                                               bbox  \
0  00003.jpg  [0.10259433962264151, 0, 0.4056603773584906, 0...   
1  00003.jpg  [0.5141509433962265, 0.009671179883945842, 0.4...   
2  00003.jpg  [0.5165094339622641, 0.5357833655705996, 0.463...   
3  00007.jpg  [0.21518987341772153, 0.005, 0.597046413502109...   
4  00007.jpg  [0.11814345991561181, 0.17333333333333334, 0.7...   

                                       label                        image_path  
0       ALL you NEED is 20 SECONDS of Insane  dataset2_2/test/images/00003.jpg  
1  COURAGE AND I PROMISE YOU something GREAT  dataset2_2/test/images/00003.jpg  
2                will come of it Benjmin Mee  dataset2_2/test/images/00003.jpg  
3                        Chalky \/n elements  dataset2_2/test/images/00007.jpg  
4                             banners Ribbon  dataset2_2/test/images/00007.jpg  


In [2]:
test_df.head(25)

,image_name,bbox,label,image_path
0,00003.jpg,"[0.10259433962264151, 0, 0.4056603773584906, 0...",ALL you NEED is 20 SECONDS of Insane,dataset2_2/test/images/00003.jpg
1,00003.jpg,"[0.5141509433962265, 0.009671179883945842, 0.4...",COURAGE AND I PROMISE YOU something GREAT,dataset2_2/test/images/00003.jpg
2,00003.jpg,"[0.5165094339622641, 0.5357833655705996, 0.463...",will come of it Benjmin Mee,dataset2_2/test/images/00003.jpg
3,00007.jpg,"[0.21518987341772153, 0.005, 0.597046413502109...",Chalky \/n elements,dataset2_2/test/images/00007.jpg
4,00007.jpg,"[0.11814345991561181, 0.17333333333333334, 0.7...",banners Ribbon,dataset2_2/test/images/00007.jpg
5,00007.jpg,"[0.23628691983122363, 0.29833333333333334, 0.4...",frames \/n try,dataset2_2/test/images/00007.jpg
6,00007.jpg,"[0.3080168776371308, 0.525, 0.4071729957805907...",FLouRiSheS,dataset2_2/test/images/00007.jpg
7,00007.jpg,"[0.11814345991561181, 0.8, 0.6497890295358649,...",and E to,dataset2_2/test/images/00007.jpg
8,00012.jpg,"[0.009510869565217392, 0.19429347826086957, 0....",Мангальные \/n блюда \/n шашлык \/n баранина г...,dataset2_2/test/images/00012.jpg
9,00012.jpg,"[0.014945652173913044, 0.5475543478260869, 0.4...",печень \/n домашняя в сетке \/n цыплёнок \/n т...,dataset2_2/test/images/00012.jpg


In [3]:
import pandas as pd
import json
import os
from pathlib import Path

def parse_train_info(csv_path):
    """
    Парсит train/info.csv с обработкой возможных ошибок
    Возвращает DataFrame с колонками: image_name, bbox, label, image_path
    """
    try:
        # Читаем CSV файл с обработкой возможных ошибок
        df = pd.read_csv(csv_path, on_bad_lines='warn')
        
        # Проверяем необходимые колонки
        required_columns = ['image_name', 'box_and_label', 'image_path']
        missing_cols = [col for col in required_columns if col not in df.columns]
        if missing_cols:
            raise ValueError(f"Отсутствуют обязательные колонки: {missing_cols}")

        # Создаем списки для новых данных
        image_names = []
        bboxes = []
        labels = []
        image_paths = []
        error_rows = []

        # Обрабатываем каждую строку
        for idx, row in df.iterrows():
            try:
                # Парсим JSON с аннотациями
                annotations = json.loads(row['box_and_label'])
                
                # Для каждого бокса создаем отдельную строку
                for ann in annotations[0]:
                    image_names.append(row['image_name'])
                    bboxes.append([
                        ann['left'],
                        ann['top'],
                        ann['width'],
                        ann['height']
                    ])
                    labels.append(ann['label'].replace('\n', ' '))
                    image_paths.append(str(Path('dataset2_2/train') / row['image_path']))
                    
            except json.JSONDecodeError as e:
                error_rows.append((idx, "JSONDecodeError", str(e)))
                continue
            except KeyError as e:
                error_rows.append((idx, "KeyError", f"Отсутствует ключ: {e}"))
                continue
            except Exception as e:
                error_rows.append((idx, type(e).__name__, str(e)))
                continue

        # Создаем DataFrame с результатами
        result_df = pd.DataFrame({
            'image_name': image_names,
            'bbox': bboxes,
            'label': labels,
            'image_path': image_paths
        })

        # Логируем ошибки
        if error_rows:
            error_df = pd.DataFrame(error_rows, columns=['row_num', 'error_type', 'message'])
            print(f"\nНайдены ошибки в {len(error_rows)} строках:")
            print(error_df)
            
            # Сохраняем ошибки в файл
            error_log_path = Path(csv_path).parent / 'parse_errors.csv'
            error_df.to_csv(error_log_path, index=False)
            print(f"\nЛог ошибок сохранен в: {error_log_path}")

        return result_df

    except Exception as e:
        print(f"\nКритическая ошибка при обработке файла: {e}")
        return None

# Основной код
if __name__ == "__main__":
    # Путь к файлу
    base_path = Path.cwd()
    train_csv_path = Path(r'dataset2_2') / 'train' / 'info.csv'
    
    # Проверяем существование файла
    if not train_csv_path.exists():
        print(f"Ошибка: Файл не найден: {train_csv_path}")
        print("Текущая структура папок:")
        for root, dirs, files in os.walk(base_path):
            print(f"{root}/")
            for f in files:
                print(f"  - {f}")
    else:
        print(f"Обрабатываем файл: {train_csv_path}")
        
        # Парсим файл
        train_df = parse_train_info(train_csv_path)
        
        if train_df is not None:
            # Выводим информацию о результате
            print("\nУспешно обработано строк:", len(train_df))
            print("\nПример данных:")
            print(train_df.head())
            
            # Сохраняем результат
            output_path = base_path / 'd2_2_train.csv'              # Стало
            train_df.to_csv(output_path, index=False)
            print(f"\nРезультат сохранен в: {output_path}")

Обрабатываем файл: dataset2_2/train/info.csv

Успешно обработано строк: 60961

Пример данных:
  image_name                                               bbox  \
0  00001.jpg  [0.025925925925925925, 0.007407407407407408, 0...   
1  00001.jpg  [0.850925925925926, 0.4703703703703704, 0.1120...   
2  00002.jpg  [0.16927083333333334, 0.015444015444015444, 0....   
3  00003.jpg  [0.10259433962264151, 0, 0.4056603773584906, 0...   
4  00003.jpg  [0.5141509433962265, 0.009671179883945842, 0.4...   

                                       label  \
0               Мы варим кофе Свежей обжарки   
1                                       мила   
2          Есть или не Есть вот в чём вопрос   
3       ALL you NEED is 20 SECONDS of Insane   
4  COURAGE AND I PROMISE YOU something GREAT   

                          image_path  
0  dataset2_2/train/images/00001.jpg  
1  dataset2_2/train/images/00001.jpg  
2  dataset2_2/train/images/00002.jpg  
3  dataset2_2/train/images/00003.jpg  
4  dataset2_2/train

In [11]:
train_df.head(25)


,image_name,bbox,label,image_path
0,00001.jpg,"[0.025925925925925925, 0.007407407407407408, 0...",Мы варим кофе Свежей обжарки,dataset2_2/train/images/00001.jpg
1,00001.jpg,"[0.850925925925926, 0.4703703703703704, 0.1120...",мила,dataset2_2/train/images/00001.jpg
2,00002.jpg,"[0.16927083333333334, 0.015444015444015444, 0....",Есть или не Есть вот в чём вопрос,dataset2_2/train/images/00002.jpg
3,00003.jpg,"[0.10259433962264151, 0, 0.4056603773584906, 0...",ALL you NEED is 20 SECONDS of Insane,dataset2_2/train/images/00003.jpg
4,00003.jpg,"[0.5141509433962265, 0.009671179883945842, 0.4...",COURAGE AND I PROMISE YOU something GREAT,dataset2_2/train/images/00003.jpg
5,00003.jpg,"[0.5165094339622641, 0.5357833655705996, 0.463...",will come of it Benjmin Mee,dataset2_2/train/images/00003.jpg
6,00004.jpg,"[0.008641975308641974, 0.0029239766081871343, ...",OLD FASHIONED HOT cocoa,dataset2_2/train/images/00004.jpg
7,00004.jpg,"[0.14938271604938272, 0.45614035087719296, 0.5...",COMBINE:,dataset2_2/train/images/00004.jpg
8,00004.jpg,"[0.4728395061728395, 0.5048732943469786, 0.222...",UNSWEETENED cocoa,dataset2_2/train/images/00004.jpg
9,00004.jpg,"[0.3012345679012346, 0.5155945419103314, 0.166...",1 3 cup,dataset2_2/train/images/00004.jpg
